In [1]:
import os
import glob
import time
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import timedelta
from PIL import Image
from ultralytics import YOLO
import torch
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from pyproj import Transformer
import cv2
import pandas as pd
import numpy as np
import json
from shapely.geometry import Polygon
from sklearn.cluster import DBSCAN

# ======================== Drawing ========================
def draw_annotations_on_image(full_image_array, slices, object_infos, output_path):
    """
    Overlays bounding boxes, masks, and lat/lon centers with:
    - Transparent masks (alpha=0.3)
    - Class/conf labels
    - Red dot + lat/lon text
    - JPEG output
    """
    import cv2
    import numpy as np
    import json
    from rasterio.transform import rowcol

    # Convert rasterio (C,H,W) to OpenCV (H,W,C)
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    overlay = img.copy()

    # Offset lookup
    slice_offsets = {tuple(slice_data["offset_xy"]): slice_data for slice_data in slices}

    for obj in object_infos:
        tile_name = obj['tile_name']
        offset_xy = (int(obj['global_x'] - obj['x_center']), int(obj['global_y'] - obj['y_center']))

        # Bounding box
        x1_global = int(obj['bbox_x1'] + offset_xy[0])
        y1_global = int(obj['bbox_y1'] + offset_xy[1])
        x2_global = int(obj['bbox_x2'] + offset_xy[0])
        y2_global = int(obj['bbox_y2'] + offset_xy[1])

        conf = obj['confidence']
        cls = obj['class']
        label = f"Cls:{cls} ({conf:.2f})"

        # Draw bounding box
        cv2.rectangle(overlay, (x1_global, y1_global), (x2_global, y2_global), (0, 255, 0), 3)

        # Label text
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 2.5
        thickness = 5
        text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
        text_origin = (x1_global, max(y1_global - 10, text_size[1]))
        cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness + 2)
        cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)

        # Mask overlay
        try:
            mask_points = np.array(json.loads(obj['mask_points']), dtype=np.int32)
            mask_points_global = mask_points + np.array([offset_xy[0], offset_xy[1]])

            if len(mask_points_global) >= 3:
                mask_img = np.zeros_like(img)
                cv2.fillPoly(mask_img, [mask_points_global], (0, 0, 255))
                cv2.polylines(mask_img, [mask_points_global], isClosed=True, color=(0, 0, 255), thickness=2)
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.7, 0)
        except:
            pass

        # Lat/Lon center overlay
        try:
            lat = float(obj['latitude'])
            lon = float(obj['longitude'])

            # Use reverse transform to get row,col (y,x)
            for slice_data in slices:
                transform = slice_data['transform']
                try:
                    row, col = rowcol(transform, lon, lat)
                    pixel_x = col + slice_data["offset_xy"][0]
                    pixel_y = row + slice_data["offset_xy"][1]
                    break
                except:
                    continue

            # Draw center point
            cv2.circle(overlay, (int(pixel_x), int(pixel_y)), 8, (0, 0, 255), -1)

            # Add lat/lon text
            ll_label = f"({lat:.5f}, {lon:.5f})"
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 3)
            cv2.putText(overlay, (int(pixel_x + 10), int(pixel_y - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        except:
            continue

    # Save as JPEG
    output_path_jpg = output_path.replace(".png", ".jpg")
    cv2.imwrite(output_path_jpg, overlay)
    print(f"📸 Annotated image saved to: {output_path_jpg}")

# ======================== Sliding ========================
from rasterio.windows import Window
from rasterio.transform import from_origin

def slide_geotiff(geotiff_path, window_size, stride):
    """
    Splits a GeoTIFF into window_size×window_size tiles with given stride,
    **and** adds extra overlapping tiles on the right/bottom so that no pixel is left out.
    """
    tiles = []
    with rasterio.open(geotiff_path) as src:
        height, width = src.height, src.width

        # 1) Compute all x-starts
        if width <= window_size:
            x_starts = [0]
        else:
            x_starts = list(range(0, width - window_size + 1, stride))
            # leftover on right?
            if x_starts[-1] + window_size < width:
                x_starts.append(width - window_size)

        # 2) Compute all y-starts
        if height <= window_size:
            y_starts = [0]
        else:
            y_starts = list(range(0, height - window_size + 1, stride))
            # leftover on bottom?
            if y_starts[-1] + window_size < height:
                y_starts.append(height - window_size)

        # 3) Build tiles for every (x,y) pair
        for y in y_starts:
            for x in x_starts:
                window = Window(x, y, window_size, window_size)
                slice_img = src.read(window=window)
                new_transform = from_origin(
                    src.transform.c + x * src.transform.a,
                    src.transform.f + y * src.transform.e,
                    src.transform.a,
                    -src.transform.e
                )
                tiles.append({
                    "img_array": slice_img,
                    "transform": new_transform,
                    "crs": src.crs,
                    "position": (y, x),
                    "offset_xy": (x, y)
                })

    return tiles


# ======================== Per-Tile Suppression ========================
from torchvision.ops import nms
def maximum_likelihood_suppression(results, conf_threshold=0.7, iou_threshold=0.5):
    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return results
    boxes = results.boxes.xyxy
    scores = results.boxes.conf
    keep = scores >= conf_threshold
    if keep.sum() == 0:
        results.boxes.data = results.boxes.data[[]]
        if results.masks and results.masks.data is not None:
            results.masks.data = results.masks.data[[]]
        return results
    keep_indices = nms(boxes[keep], scores[keep], iou_threshold)
    indices_to_keep = torch.nonzero(keep).squeeze(1)[keep_indices]
    results.boxes.data = results.boxes.data[indices_to_keep]
    if results.masks and results.masks.data is not None:
        results.masks.data = results.masks.data[indices_to_keep]
    return results

# ======================== Per-Tile Extraction ========================
def overlay_segmentation_on_array(results, transform, img_shape, tif_image, offset_xy):
    object_data = []
    masks_out = []

    if results.masks is None or results.boxes is None or results.boxes.data is None:
        return object_data, masks_out

    for idx, (mask, box, cls, conf) in enumerate(zip(
        results.masks.xy, results.boxes.xyxy, results.boxes.cls, results.boxes.conf)):

        cls = int(cls.item())
        conf = float(conf.item())
        x1, y1, x2, y2 = map(float, box.tolist())

        mask_pixel_coords = np.array(mask.tolist(), dtype=np.float32)
        mask_gis_coords = []
        for px, py in mask_pixel_coords:
            try:
                lon, lat = rasterio.transform.xy(transform, py, px)
                mask_gis_coords.append([lon, lat])
            except:
                continue

        pixel_count = len(mask_pixel_coords)
        area = pixel_count * (transform.a * abs(transform.e))
        local_cx = (x1 + x2) / 2
        local_cy = (y1 + y2) / 2
        global_x = offset_xy[0] + local_cx
        global_y = offset_xy[1] + local_cy
        lon_center, lat_center = rasterio.transform.xy(transform, local_cy, local_cx)
        lon_x1, lat_y1 = rasterio.transform.xy(transform, y1, x1)
        lon_x2, lat_y2 = rasterio.transform.xy(transform, y2, x2)
        bbox_width_geo = abs(lon_x2 - lon_x1)
        bbox_height_geo = abs(lat_y2 - lat_y1)

        object_data.append({
            "latitude": lat_center,
            "longitude": lon_center,
            "bbox_lat_y1": lat_y1,
            "bbox_lon_x1": lon_x1,
            "bbox_lat_y2": lat_y2,
            "bbox_lon_x2": lon_x2,
            "bbox_width_geo": bbox_width_geo,
            "bbox_height_geo": bbox_height_geo,
            "global_x": global_x,
            "global_y": global_y,
            "x_center": local_cx,
            "y_center": local_cy,
            "bbox_x1": x1,
            "bbox_y1": y1,
            "bbox_x2": x2,
            "bbox_y2": y2,
            "bbox_width": x2 - x1,
            "bbox_height": y2 - y1,
            "area_geo": area,
            "pixel_count": pixel_count,
            "class": cls,
            "confidence": conf,
            "tile_name": tif_image,
            "mask_index": idx,
            "mask_points": json.dumps(mask_pixel_coords.tolist()),
            "mask_points_gis": json.dumps(mask_gis_coords)
        })

        masks_out.append(mask_pixel_coords)

    return object_data, masks_out


# ======================== Full Pipeline Runner ========================
def run_inference(folder_path, model_path, final_csv_path, window_size=3000, stride=3000, batch_size=16):
    model = YOLO(model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    print(f"Using device: {device}")

    # GPU Warmup
    _ = model([Image.new('RGB', (window_size, window_size))], device=device, verbose=False)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    accumulate_object_infos = []
    for tif_image in tqdm(tif_images, desc="Overall Progress"):
        slices = slide_geotiff(tif_image, window_size, stride)
        for batch in [slices[i:i + batch_size] for i in range(0, len(slices), batch_size)]:
            pil_imgs_batch = [Image.fromarray(np.moveaxis(s["img_array"][:3], 0, -1)) for s in batch]
            with torch.inference_mode():
                results_batch = model(pil_imgs_batch, device=device, batch=batch_size, verbose=False)
            for slice_data, results in zip(batch, results_batch):
                results = maximum_likelihood_suppression(results)
                object_info, _ = overlay_segmentation_on_array(results, slice_data["transform"], None, os.path.basename(tif_image), slice_data["offset_xy"])
                accumulate_object_infos.extend(object_info)

    pd.DataFrame(accumulate_object_infos).to_csv(final_csv_path, index=False)
    print(f"✅ Final results saved to {final_csv_path}")


# ======================== Visualization ========================
def run_viz_sup(folder_path, csv_sup="annotated_output_test/final_sup.csv", window_size=3000, stride=3000):
    suppressed_infos = pd.read_csv(csv_sup)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    for tif_image in tif_images:
        with rasterio.open(tif_image) as src:
            full_image_array = src.read()
        image_name = os.path.splitext(os.path.basename(tif_image))[0]

        # Correct way to filter using DataFrame
        matched_objs = suppressed_infos[suppressed_infos['tile_name'].str.split(',').str[0] == os.path.basename(tif_image)]
        # Convert filtered DataFrame to dict list
        matched_objs = matched_objs.to_dict(orient='records')

        draw_annotations_on_image(full_image_array, slide_geotiff(tif_image, window_size, stride),
                                  matched_objs,
                                  f"annotated_output_test_1/{image_name}_annotated.jpg")

# ======================== making ractangles ========================

import pandas as pd
import json
from shapely.geometry import Polygon
import numpy as np

def shrink_rect_corners(corners, shrink_pixels=2):
    """
    Moves each corner towards the centroid by `shrink_pixels`.
    """
    arr = np.array(corners)
    centroid = arr.mean(axis=0)
    vectors = centroid - arr
    lengths = np.linalg.norm(vectors, axis=1)
    # Avoid division by zero
    move_vectors = np.where(lengths[:, np.newaxis] > 0, vectors / lengths[:, np.newaxis], 0)
    new_corners = arr + move_vectors * shrink_pixels
    return new_corners.tolist()

def make_rectangles(input_csv: str, output_csv: str, shrink_pixels=2):
    """
    Same as before, but rectangle is slightly shrunken.
    """
    df = pd.read_csv(input_csv)

    rect_points = []
    rect_bboxes = []
    rect_areas = []
    rect_centers = []

    for raw in df['mask_points_gis']:
        pts = json.loads(raw)
        poly = Polygon(pts)
        rect = poly.minimum_rotated_rectangle
        coords = list(rect.exterior.coords)[:-1]
        corners = [[x, y] for x, y in coords]

        # Shrink rectangle corners toward centroid
        shrunk_corners = shrink_rect_corners(corners, shrink_pixels)

        rect_points.append(json.dumps(shrunk_corners))
        rect_bboxes.append(tuple(tuple(c) for c in shrunk_corners))
        shrunk_poly = Polygon(shrunk_corners)
        rect_areas.append(shrunk_poly.area)
        cx, cy = shrunk_poly.centroid.x, shrunk_poly.centroid.y
        rect_centers.append((cx, cy))

    df['rect_mask_points'] = rect_points
    df['rect_bbox']        = rect_bboxes
    df['rect_area']        = rect_areas
    df['rect_x_center']    = [c[0] for c in rect_centers]
    df['rect_y_center']    = [c[1] for c in rect_centers]

    df.to_csv(output_csv, index=False)
    print(f"Written {len(df)} rows with *shrunk* rectangle info to {output_csv!r}")

# Example usage:
# make_rectangles("input.csv", "rects.csv", shrink_pixels=2)

# make_rectangles(input_csv="annotated_output_test/final.csv", output_csv="annotated_output_test/final_rect.csv")


# ======================== draw_annotations_with_rectangles ========================
import os
import glob
import json
import ast
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.windows import Window
from rasterio.transform import from_origin

def draw_annotations_with_rectangles(full_image_array, slices, object_infos, output_path):
    """
    Overlays:
      • Red 5px boundaries around each sliding window
      • YOLO bounding boxes (green)
      • YOLO masks (transparent red fill + outline)
      • Axis-aligned rectangle from 'rect_mask_points' (blue outline)
      • Rectangle centroid (blue dot)
      • Rectangle area & bbox text (blue)
      • Class/conf labels
      • JPEG output
    """
    # 1) Prepare base image (BGR)
    img = np.moveaxis(full_image_array[:3], 0, -1)
    img = np.clip((img / img.max()) * 255, 0, 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    overlay = img.copy()

    # 2) Draw sliding-window boundaries in red
    for sl in slices:
        x_off, y_off = sl['offset_xy']
        h = sl['img_array'].shape[1]  # window height
        w = sl['img_array'].shape[2]  # window width
        cv2.rectangle(overlay,
                      (x_off, y_off),
                      (x_off + w, y_off + h),
                      (0, 0, 255),  # red
                      5)            # thickness

    # 3) Styles for later
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs_label, th_label = 2.5, 5
    fs_area,  th_area  = 1.5, 3
    fs_box,   th_box   = 1.2, 2

    # 4) Draw each object’s annotations
    for obj in object_infos:
        # compute tile→full-image offset
        ox = int(obj['global_x'] - obj['x_center'])
        oy = int(obj['global_y'] - obj['y_center'])
        offset = np.array([ox, oy], dtype=np.int32)

        # # — YOLO bbox & label —
        # x1, y1 = int(obj['bbox_x1'] + ox), int(obj['bbox_y1'] + oy)
        # x2, y2 = int(obj['bbox_x2'] + ox), int(obj['bbox_y2'] + oy)
        # cv2.rectangle(overlay, (x1, y1), (x2, y2), (0,255,0), 3)
        # label = f"Cls:{obj['class']} ({obj['confidence']:.2f})"
        # ts = cv2.getTextSize(label, font, fs_label, th_label)[0]
        # origin = (x1, max(y1 - 10, ts[1]))
        # cv2.putText(overlay, label, origin, font, fs_label, (0,0,0), th_label+2)
        # cv2.putText(overlay, label, origin, font, fs_label, (255,255,255), th_label)

        # — YOLO mask —
        try:
            pts = np.array(json.loads(obj['mask_points']), dtype=np.int32) + offset
            mask_img = np.zeros_like(img)
            cv2.fillPoly(mask_img, [pts], (0,0,255))
            cv2.polylines(mask_img, [pts], True, (0,0,255), 2)
            overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.3, 0)
        except:
            pass

        # — Axis-aligned rectangle & info —
        try:
            # corners in GIS coords → pixel coords
            rect_gis = json.loads(obj['rect_mask_points'])
            pix_pts = []
            for X, Y in rect_gis:
                for sl in slices:
                    try:
                        r, c = rowcol(sl['transform'], X, Y)
                        px = c + sl['offset_xy'][0]
                        py = r + sl['offset_xy'][1]
                        pix_pts.append([px, py])
                        break
                    except:
                        continue
            pix_pts = np.array(pix_pts, dtype=np.int32)

            # draw rectangle outline
            cv2.polylines(overlay, [pix_pts], True, (255,0,0), 3)

            # draw centroid
            Xc, Yc = float(obj['rect_x_center']), float(obj['rect_y_center'])
            for sl in slices:
                try:
                    rr, cc = rowcol(sl['transform'], Xc, Yc)
                    cx = int(cc + sl['offset_xy'][0])
                    cy = int(rr + sl['offset_xy'][1])
                    break
                except:
                    continue
            cv2.circle(overlay, (cx, cy), 6, (255,0,0), -1)

            # annotate area
            area_txt = f"A:{obj['rect_area']:.1f}"
            pt_area = (pix_pts[0][0], pix_pts[0][1] - 10)
            cv2.putText(overlay, area_txt, pt_area, font, fs_area, (255,0,0), th_area)

            # # annotate bbox tuple
            # bbox = ast.literal_eval(obj['rect_bbox'])
            # bbox_txt = f"BB:{bbox[0]:.0f},{bbox[1]:.0f},{bbox[2]:.0f},{bbox[3]:.0f}"
            # br = pix_pts[2]
            # pt_box = (br[0] - 200, br[1] + 30)
            # cv2.putText(overlay, bbox_txt, pt_box, font, fs_box, (255,0,0), th_box)

        except:
            pass

    # 5) Save as JPEG
    out_jpg = output_path.replace(".png", ".jpg")
    cv2.imwrite(out_jpg, overlay)
    print(f"📸 Saved annotated image with rectangles and window grid to: {out_jpg}")


def run_viz_rect(folder_path, csv_sup="annotated_output_test/final_rect.csv",
                 window_size=3000, stride=3000):
    suppressed_infos = pd.read_csv(csv_sup)
    tif_images = glob.glob(os.path.join(folder_path, '**', '*.tif'), recursive=True)
    for tif_image in tif_images:
        with rasterio.open(tif_image) as src:
            full_image_array = src.read()
        image_name = os.path.splitext(os.path.basename(tif_image))[0]

        matched = suppressed_infos[
            suppressed_infos['tile_name'].str.split(',').str[0] == os.path.basename(tif_image)
        ]
        objs = matched.to_dict(orient='records')

        draw_annotations_with_rectangles(
            full_image_array,
            slide_geotiff(tif_image, window_size, stride),
            objs,
            f"annotated_output_test_1/{image_name}_rect_annotated.jpg"
        )


# run_viz_rect(folder_path="C:/Users/mza0288/Documents/Code/test_sample", csv_sup="annotated_output_test_1/final_rect_merged.csv")


# ======================== merge_adjacent_masks_debug ========================
import pandas as pd
import ast
from shapely.geometry import Polygon, LineString
from shapely.ops import unary_union
import itertools

def polygons_are_adjacent(poly1, poly2, threshold):
    edges1 = [LineString([poly1.exterior.coords[i], poly1.exterior.coords[i+1]]) 
              for i in range(len(poly1.exterior.coords)-1)]
    edges2 = [LineString([poly2.exterior.coords[i], poly2.exterior.coords[i+1]]) 
              for i in range(len(poly2.exterior.coords)-1)]
    
    # Check each edge against edges of another polygon
    for edge1, edge2 in itertools.product(edges1, edges2):
        if edge1.distance(edge2) <= threshold:
            return True
    return False

def merge_adjacent_masks(input_csv, output_csv, distance_threshold=2.0):
    df = pd.read_csv(input_csv)
    df['polygon'] = df['rect_mask_points'].apply(lambda x: Polygon(ast.literal_eval(x)))
    polygons = df['polygon'].tolist()

    n = len(polygons)
    adjacency_list = {i: set() for i in range(n)}

    # Build adjacency list carefully based on edge proximity
    for i, j in itertools.combinations(range(n), 2):
        if polygons_are_adjacent(polygons[i], polygons[j], distance_threshold):
            adjacency_list[i].add(j)
            adjacency_list[j].add(i)

    # Group polygons using adjacency list
    visited = set()
    merged_groups = []

    def dfs(current_idx, group):
        visited.add(current_idx)
        group.add(current_idx)
        for neighbor in adjacency_list[current_idx]:
            if neighbor not in visited:
                dfs(neighbor, group)

    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            merged_groups.append(group)

    merged_data = []

    for group_indices in merged_groups:
        merge_polys = [polygons[i] for i in group_indices]
        merged_geometry = unary_union(merge_polys)

        if merged_geometry.geom_type == 'MultiPolygon':
            merged_geometry = list(merged_geometry.geoms)
        else:
            merged_geometry = [merged_geometry]

        for single_geom in merged_geometry:
            group_rows = df.loc[list(group_indices)]
            highest_confidence_row = group_rows.loc[group_rows['confidence'].idxmax()].copy()

            minx, miny, maxx, maxy = single_geom.bounds
            highest_confidence_row['mask_points_gis'] = str(list(single_geom.exterior.coords))
            highest_confidence_row['bbox_lon_x1'], highest_confidence_row['bbox_lat_y1'] = minx, miny
            highest_confidence_row['bbox_lon_x2'], highest_confidence_row['bbox_lat_y2'] = maxx, maxy
            highest_confidence_row['bbox_width_geo'] = maxx - minx
            highest_confidence_row['bbox_height_geo'] = maxy - miny
            highest_confidence_row['rect_area'] = single_geom.area
            centroid = single_geom.centroid
            highest_confidence_row['rect_x_center'], highest_confidence_row['rect_y_center'] = centroid.x, centroid.y
            highest_confidence_row['latitude'], highest_confidence_row['longitude'] = centroid.y, centroid.x
            highest_confidence_row['bbox_width'], highest_confidence_row['bbox_height'] = maxx - minx, maxy - miny

            merged_data.append(highest_confidence_row)

    final_df = pd.DataFrame(merged_data)

    final_df.drop(columns=['polygon'], inplace=True)

    final_df.to_csv(output_csv, index=False)

# Example usage:
# merge_adjacent_masks('annotated_output_test_1/final_rect.csv', 'annotated_output_test_1/final_rect_merged.csv', distance_threshold=0.2)


In [3]:
# # folder_path="D:/Data/al_30cm_2021_transfer_1_of_2/31085",
# # folder_path="C:/Users/mza0288/Documents/Code/geotiff_sample",
run_inference(
    folder_path="D:/Data/",
    model_path="C:/Users/mza0288/Documents/Code/runs/segment/yolo11s-500-sgd-04082025/weights/best.pt",
    final_csv_path="CSV_Yolo/final.csv",
    window_size=3000, stride=3000, batch_size=32
)

Using device: cuda


Overall Progress: 100%|█████████████████████████████████████████████████████████| 3480/3480 [44:00:15<00:00, 45.52s/it]


✅ Final results saved to CSV_Yolo/final.csv


In [408]:
import pandas as pd
import json
from shapely.geometry import Polygon, MultiPolygon, LineString
from shapely.strtree import STRtree
from shapely.ops import unary_union

def edges_from_polygon(poly):
    coords = list(poly.exterior.coords)
    return [LineString([coords[i], coords[i+1]]) for i in range(len(coords)-1)]

def polygons_edges_are_close(poly1, poly2, threshold=2):
    edges1 = edges_from_polygon(poly1)
    edges2 = edges_from_polygon(poly2)
    for e1 in edges1:
        for e2 in edges2:
            if e1.distance(e2) < threshold:
                return True
    return False

def merge_masks_by_edge_proximity(input_csv, output_csv, edge_threshold=2.0, fill_gap=True):
    df0 = pd.read_csv(input_csv)

    polygons = []
    df_rows = []
    for idx, x in enumerate(df0['mask_points_gis']):
        p = Polygon(json.loads(x))
        if not p.is_valid:
            p = p.buffer(0)
        # Handle MultiPolygon by adding each part separately, with the same row data
        if isinstance(p, MultiPolygon):
            for subp in p.geoms:
                if subp.is_empty or not subp.is_valid:
                    continue
                polygons.append(subp)
                df_rows.append(df0.iloc[idx].copy())
        elif p.is_empty or not p.is_valid:
            continue
        else:
            polygons.append(p)
            df_rows.append(df0.iloc[idx].copy())
    df = pd.DataFrame(df_rows).reset_index(drop=True)
    n = len(polygons)

    # Build spatial index (STRtree)
    tree = STRtree(polygons)
    def get_candidates(poly):
        result = tree.query(poly.buffer(edge_threshold))
        # Handle both geometries and integer indices
        if hasattr(result[0], 'wkt'):
            wkt_to_idx = {p.wkt: i for i, p in enumerate(polygons)}
            return [wkt_to_idx.get(p.wkt, None) for p in result]
        else:
            return list(result)

    adjacency = {i: set() for i in range(n)}
    for i, poly in enumerate(polygons):
        candidates = get_candidates(poly)
        for j in candidates:
            if j is None or i >= j:
                continue
            if polygons_edges_are_close(poly, polygons[j], edge_threshold):
                adjacency[i].add(j)
                adjacency[j].add(i)

    # DFS for connected groups
    visited = set()
    groups = []
    def dfs(idx, group):
        group.add(idx)
        visited.add(idx)
        for nb in adjacency[idx]:
            if nb not in visited:
                dfs(nb, group)
    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            groups.append(group)

    merged_data = []
    for group in groups:
        group = list(group)
        try:
            polys_to_merge = [polygons[i] for i in group]
            if fill_gap:
                # Buffer out, union, buffer back in to fill small gaps
                buffered = [p.buffer(edge_threshold) for p in polys_to_merge]
                merged_poly = unary_union(buffered)
                merged_poly = merged_poly.buffer(-edge_threshold)
            else:
                merged_poly = unary_union(polys_to_merge)
        except Exception as e:
            print(f"⚠️ Error merging group {group}: {e}")
            continue
        if merged_poly.is_empty:
            continue
        if merged_poly.geom_type == "MultiPolygon":
            polys = list(merged_poly.geoms)
        else:
            polys = [merged_poly]

        for poly in polys:
            if poly.is_empty or not poly.is_valid:
                continue
            group_rows = df.loc[group]
            highest_confidence_row = group_rows.loc[group_rows['confidence'].idxmax()].copy()
            minx, miny, maxx, maxy = poly.bounds
            mask_points = list(poly.exterior.coords)
            highest_confidence_row['mask_points_gis'] = json.dumps(mask_points)
            highest_confidence_row['bbox_lon_x1'], highest_confidence_row['bbox_lat_y1'] = minx, miny
            highest_confidence_row['bbox_lon_x2'], highest_confidence_row['bbox_lat_y2'] = maxx, maxy
            highest_confidence_row['bbox_width_geo'] = maxx - minx
            highest_confidence_row['bbox_height_geo'] = maxy - miny
            highest_confidence_row['rect_area'] = poly.area
            centroid = poly.centroid
            highest_confidence_row['rect_x_center'], highest_confidence_row['rect_y_center'] = centroid.x, centroid.y
            highest_confidence_row['latitude'], highest_confidence_row['longitude'] = centroid.y, centroid.x
            highest_confidence_row['bbox_width'], highest_confidence_row['bbox_height'] = maxx - minx, maxy - miny
            merged_data.append(highest_confidence_row)

    final_df = pd.DataFrame(merged_data)
    final_df.to_csv(output_csv, index=False)
    print(f"Wrote merged results to {output_csv}")

# Example usage:
merge_masks_by_edge_proximity("annotated_output_test_1/final.csv",
                              "annotated_output_test_1/final_1.csv",
                              edge_threshold=1.7,
                              fill_gap=True)


Wrote merged results to annotated_output_test_1/final_1.csv


In [409]:
make_rectangles(input_csv="annotated_output_test_1/final.csv", output_csv="annotated_output_test_1/final_rect.csv",shrink_pixels=3)

Written 3490 rows with *shrunk* rectangle info to 'annotated_output_test_1/final_rect.csv'


In [410]:
import pandas as pd
import ast
from shapely.geometry import Polygon, LineString
from shapely.ops import unary_union
import itertools
import numpy as np

def get_middle_edge_line(poly):
    coords = list(poly.exterior.coords)
    edges = [(coords[i], coords[i+1]) for i in range(len(coords)-1)]
    edge_lengths = [LineString([p1, p2]).length for p1, p2 in edges]
    short_edge_indices = np.argsort(edge_lengths)[:2]
    midpoints = [
        LineString(edges[i]).interpolate(0.5, normalized=True) for i in short_edge_indices
    ]
    return LineString(midpoints)

def lines_are_close(line1, line2, threshold):
    return line1.distance(line2) <= threshold

def merge_adjacent_masks(input_csv, output_csv, distance_threshold=2.0):
    df = pd.read_csv(input_csv)
    df['polygon'] = df['rect_mask_points'].apply(lambda x: Polygon(ast.literal_eval(x)))
    polygons = df['polygon'].tolist()

    n = len(polygons)
    adjacency_list = {i: set() for i in range(n)}

    # Build adjacency list based on proximity of central lines
    center_lines = [get_middle_edge_line(p) for p in polygons]

    for i, j in itertools.combinations(range(n), 2):
        if lines_are_close(center_lines[i], center_lines[j], distance_threshold):
            adjacency_list[i].add(j)
            adjacency_list[j].add(i)

    visited = set()
    merged_groups = []

    def dfs(current_idx, group):
        visited.add(current_idx)
        group.add(current_idx)
        for neighbor in adjacency_list[current_idx]:
            if neighbor not in visited:
                dfs(neighbor, group)

    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            merged_groups.append(group)

    merged_data = []

    for group_indices in merged_groups:
        merge_polys = [polygons[i] for i in group_indices]
        merged_geometry = unary_union(merge_polys)

        if merged_geometry.is_empty:
            continue  # skip invalid
        if merged_geometry.geom_type == 'MultiPolygon':
            polygons_to_save = list(merged_geometry.geoms)
        elif merged_geometry.geom_type == 'Polygon':
            polygons_to_save = [merged_geometry]
        else:
            continue  # skip unexpected geometry

        for single_geom in polygons_to_save:
            if single_geom.is_empty or not single_geom.is_valid:
                continue
            group_rows = df.loc[list(group_indices)]
            highest_confidence_row = group_rows.loc[group_rows['confidence'].idxmax()].copy()
            minx, miny, maxx, maxy = single_geom.bounds
            highest_confidence_row['mask_points_gis'] = str(list(single_geom.exterior.coords))
            highest_confidence_row['bbox_lon_x1'], highest_confidence_row['bbox_lat_y1'] = minx, miny
            highest_confidence_row['bbox_lon_x2'], highest_confidence_row['bbox_lat_y2'] = maxx, maxy
            highest_confidence_row['bbox_width_geo'] = maxx - minx
            highest_confidence_row['bbox_height_geo'] = maxy - miny
            highest_confidence_row['rect_area'] = single_geom.area
            centroid = single_geom.centroid
            highest_confidence_row['rect_x_center'], highest_confidence_row['rect_y_center'] = centroid.x, centroid.y
            highest_confidence_row['latitude'], highest_confidence_row['longitude'] = centroid.y, centroid.x
            highest_confidence_row['bbox_width'], highest_confidence_row['bbox_height'] = maxx - minx, maxy - miny

            merged_data.append(highest_confidence_row)

    final_df = pd.DataFrame(merged_data)
    if 'polygon' in final_df.columns:
        final_df.drop(columns=['polygon'], inplace=True)
    final_df.to_csv(output_csv, index=False)
    print(f"✅ Wrote merged results to {output_csv}")

# Example usage:
merge_adjacent_masks(
    'annotated_output_test_1/final_rect.csv',
    'annotated_output_test_1/final_rect_merged.csv',
    distance_threshold=1.0
)

✅ Wrote merged results to annotated_output_test_1/final_rect_merged.csv


In [411]:



#---------------------------------removing multiple overlaping bbox--------------------
import pandas as pd
import ast
from shapely.geometry import Polygon

def merge_rect_masks(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    polygons = [Polygon(ast.literal_eval(x)) for x in df['rect_mask_points']]
    n = len(polygons)
    areas = [poly.area for poly in polygons]

    # Build overlap graph (only real overlaps, not touches)
    overlap_graph = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(i+1, n):
            if (
                polygons[i].intersects(polygons[j])
                and polygons[i].intersection(polygons[j]).area > 0
            ):
                overlap_graph[i].add(j)
                overlap_graph[j].add(i)

    # Find all overlap groups (connected components)
    groups = []
    visited = set()
    def dfs(idx, group):
        group.add(idx)
        visited.add(idx)
        for nb in overlap_graph[idx]:
            if nb not in visited:
                dfs(nb, group)
    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            groups.append(group)

    # For each group, keep only the largest by area
    keep_indices = set()
    for group in groups:
        group = list(group)
        group_areas = [areas[i] for i in group]
        idx_largest = group[np.argmax(group_areas)]
        keep_indices.add(idx_largest)

    filtered_df = df.iloc[list(keep_indices)].copy()
    filtered_df.to_csv(output_csv, index=False)
    print(f"Kept {len(filtered_df)} polygons out of {n}.")

# Example usage:
# merge_adjacent_masks('final_rect.csv', 'final_rect_merged.csv')


merge_rect_masks('annotated_output_test_1/final_rect_merged.csv', 'annotated_output_test_1/final_rect_merged_1.csv')


Kept 2239 polygons out of 2687.


In [412]:
# run_viz_rect(folder_path="C:/Users/mza0288/Documents/Code/test_sample", csv_sup="annotated_output_test_1/final_rect_merged.csv", window_size=3000, stride=3000)

In [413]:
import pandas as pd
import json
from shapely.geometry import Polygon, MultiPolygon, LineString, Point
from shapely.strtree import STRtree
from shapely.ops import unary_union

def edges_from_polygon(poly):
    coords = list(poly.exterior.coords)
    return [LineString([coords[i], coords[i+1]]) for i in range(len(coords)-1)]

def polygons_edges_are_close(poly1, poly2, threshold=2):
    edges1 = edges_from_polygon(poly1)
    edges2 = edges_from_polygon(poly2)
    for e1 in edges1:
        for e2 in edges2:
            if e1.distance(e2) < threshold:
                return True
    return False

def furthest_four_points(all_points):
    """Given a list of [x, y], return the 4 points forming the minimum bounding rectangle."""
    # Use convex hull, then minimum rotated rectangle
    poly = Polygon(all_points)
    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_empty or not poly.is_valid:
        # fallback to min/max bounds if geometry invalid
        xs, ys = zip(*all_points)
        minx, maxx, miny, maxy = min(xs), max(xs), min(ys), max(ys)
        return [[minx, miny], [minx, maxy], [maxx, maxy], [maxx, miny]]
    mrect = poly.minimum_rotated_rectangle
    coords = list(mrect.exterior.coords)[:-1]  # Exclude closing point
    return [list(pt) for pt in coords]

def merge_masks_by_edge_proximity(input_csv, output_csv, edge_threshold=2.0, fill_gap=True):
    df0 = pd.read_csv(input_csv)

    polygons = []
    df_rows = []
    rect_points_list = []
    for idx, x in enumerate(df0['rect_mask_points']):
        # rect_mask_points is a JSON string of 4 points
        rect_points = json.loads(x)
        poly = Polygon(rect_points)
        if not poly.is_valid:
            poly = poly.buffer(0)
        # Handle MultiPolygon by adding each part separately, with the same row data
        if isinstance(poly, MultiPolygon):
            for subp in poly.geoms:
                if subp.is_empty or not subp.is_valid:
                    continue
                polygons.append(subp)
                df_rows.append(df0.iloc[idx].copy())
                rect_points_list.append(list(subp.exterior.coords)[:-1])
        elif poly.is_empty or not poly.is_valid:
            continue
        else:
            polygons.append(poly)
            df_rows.append(df0.iloc[idx].copy())
            rect_points_list.append(rect_points)
    df = pd.DataFrame(df_rows).reset_index(drop=True)
    n = len(polygons)

    # Build spatial index (STRtree)
    tree = STRtree(polygons)
    def get_candidates(poly):
        result = tree.query(poly.buffer(edge_threshold))
        if hasattr(result[0], 'wkt'):
            wkt_to_idx = {p.wkt: i for i, p in enumerate(polygons)}
            return [wkt_to_idx.get(p.wkt, None) for p in result]
        else:
            return list(result)

    adjacency = {i: set() for i in range(n)}
    for i, poly in enumerate(polygons):
        candidates = get_candidates(poly)
        for j in candidates:
            if j is None or i >= j:
                continue
            if polygons_edges_are_close(poly, polygons[j], edge_threshold):
                adjacency[i].add(j)
                adjacency[j].add(i)

    # DFS for connected groups
    visited = set()
    groups = []
    def dfs(idx, group):
        group.add(idx)
        visited.add(idx)
        for nb in adjacency[idx]:
            if nb not in visited:
                dfs(nb, group)
    for idx in range(n):
        if idx not in visited:
            group = set()
            dfs(idx, group)
            groups.append(group)

    merged_data = []
    for group in groups:
        group = list(group)
        try:
            polys_to_merge = [polygons[i] for i in group]
            group_rect_points = [pt for i in group for pt in rect_points_list[i]]
            if fill_gap:
                buffered = [p.buffer(edge_threshold) for p in polys_to_merge]
                merged_poly = unary_union(buffered).buffer(-edge_threshold)
            else:
                merged_poly = unary_union(polys_to_merge)
        except Exception as e:
            print(f"⚠️ Error merging group {group}: {e}")
            continue
        if merged_poly.is_empty:
            continue
        if merged_poly.geom_type == "MultiPolygon":
            polys = list(merged_poly.geoms)
        else:
            polys = [merged_poly]

        for poly in polys:
            if poly.is_empty or not poly.is_valid:
                continue
            group_rows = df.loc[group]
            highest_confidence_row = group_rows.loc[group_rows['confidence'].idxmax()].copy()
            minx, miny, maxx, maxy = poly.bounds
            mask_points = list(poly.exterior.coords)
            # Use the furthest 4 points for the merged rectangle
            merged_rect_points = furthest_four_points(group_rect_points)
            highest_confidence_row['rect_mask_points'] = json.dumps(merged_rect_points)
            highest_confidence_row['rect_bbox'] = str(tuple(tuple(pt) for pt in merged_rect_points))
            highest_confidence_row['rect_area'] = Polygon(merged_rect_points).area
            highest_confidence_row['rect_x_center'] = Polygon(merged_rect_points).centroid.x
            highest_confidence_row['rect_y_center'] = Polygon(merged_rect_points).centroid.y
            # Keep other fields consistent with previous logic (based on mask)
            highest_confidence_row['mask_points_gis'] = json.dumps(mask_points)
            highest_confidence_row['bbox_lon_x1'], highest_confidence_row['bbox_lat_y1'] = minx, miny
            highest_confidence_row['bbox_lon_x2'], highest_confidence_row['bbox_lat_y2'] = maxx, maxy
            highest_confidence_row['bbox_width_geo'] = maxx - minx
            highest_confidence_row['bbox_height_geo'] = maxy - miny
            centroid = poly.centroid
            highest_confidence_row['rect_x_center'], highest_confidence_row['rect_y_center'] = centroid.x, centroid.y
            highest_confidence_row['latitude'], highest_confidence_row['longitude'] = centroid.y, centroid.x
            highest_confidence_row['bbox_width'], highest_confidence_row['bbox_height'] = maxx - minx, maxy - miny
            merged_data.append(highest_confidence_row)

    final_df = pd.DataFrame(merged_data)
    final_df.to_csv(output_csv, index=False)
    print(f"Wrote merged results to {output_csv}")

# # Example usage:
# merge_masks_by_edge_proximity("annotated_output_test_1/final_rect_merged_1.csv",
#                               "annotated_output_test_1/final_rect_merged_1m.csv",
#                               edge_threshold=2.0,
#                               fill_gap=True)


In [414]:
import os
import json
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, mapping
import fiona

def save_bbox_shapefile_from_csv(csv_path, output_dir, reference_crs="EPSG:26916"):
    """
    Generates a shapefile using bounding box polygons from the CSV column 'rect_bbox'.
    """

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    shapefile_path = os.path.join(output_dir, "bbox_shapes.shp")

    # Load CSV
    df = pd.read_csv(csv_path)

    # Define schema for DBF fields
    schema = {
        'geometry': 'Polygon',
        'properties': {
            'latitude': 'float',
            'longitude': 'float',
            'class': 'int',
            'confidence': 'float',
            'tile_name': 'str:50',
            'area_geo': 'float'
        }
    }

    # Open Fiona shapefile writer
    with fiona.open(
        shapefile_path,
        mode='w',
        driver='ESRI Shapefile',
        crs=reference_crs,
        schema=schema
    ) as shp:
        for idx, row in df.iterrows():
            try:
                # Parse rect_bbox as polygon coordinates
                bbox_coords = json.loads(row['rect_bbox'].replace('(', '[').replace(')', ']'))
                
                # Validate coordinates
                if not all(np.isfinite([coord for point in bbox_coords for coord in point])):
                    continue

                # Create polygon directly from rect_bbox
                poly = Polygon(bbox_coords)

                if not poly.is_valid or poly.is_empty:
                    continue

                # Safely encode string field
                tile = str(row.get('tile_name', ''))[:50].encode('ascii', 'ignore').decode()

                # Write feature
                shp.write({
                    'geometry': mapping(poly),
                    'properties': {
                        'latitude': float(row.get('latitude', 0)),
                        'longitude': float(row.get('longitude', 0)),
                        'class': int(row.get('class', 0)),
                        'confidence': float(row.get('confidence', 0)),
                        'tile_name': tile,
                        'area_geo': float(row.get('area_geo', 0))
                    }
                })

            except Exception as e:
                print(f"⚠️ Skipped row {idx} due to error: {e}")

    print(f"\n✅ Shapefile successfully written to: {shapefile_path}")


In [415]:
save_bbox_shapefile_from_csv(
    csv_path="annotated_output_test_1/final_rect.csv",
    output_dir="annotated_output_test_1/annotated_output_box"
)
save_bbox_shapefile_from_csv(
    csv_path="annotated_output_test_1/final_rect_merged.csv",
    output_dir="annotated_output_test_1/annotated_output_box_m"
)

save_bbox_shapefile_from_csv(
    csv_path="annotated_output_test_1/final_rect_merged_1.csv",
    output_dir="annotated_output_test_1/annotated_output_box_full"
)


✅ Shapefile successfully written to: annotated_output_test_1/annotated_output_box\bbox_shapes.shp

✅ Shapefile successfully written to: annotated_output_test_1/annotated_output_box_m\bbox_shapes.shp

✅ Shapefile successfully written to: annotated_output_test_1/annotated_output_box_full\bbox_shapes.shp


In [89]:
import os
import json
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, mapping
import fiona

def save_shapefile_from_csv(csv_path, output_dir, reference_crs="EPSG:26916"):
    """
    Generates a shapefile using polygons from the 'rect_mask_points' column.
    """

    os.makedirs(output_dir, exist_ok=True)
    shapefile_path = os.path.join(output_dir, "bbox_shapes.shp")

    df = pd.read_csv(csv_path)

    # if 'rect_mask_points' not in df.columns:
    #     raise ValueError("CSV is missing required column 'rect_mask_points'")

    schema = {
        'geometry': 'Polygon',
        'properties': {
            'latitude': 'float',
            'longitude': 'float',
            'class': 'int',
            'confidence': 'float',
            'tile_name': 'str:50',
            'area_geo': 'float'
        }
    }

    with fiona.open(
        shapefile_path,
        mode='w',
        driver='ESRI Shapefile',
        crs=reference_crs,
        schema=schema
    ) as shp:
        for idx, row in df.iterrows():
            try:
                rect_coords = json.loads(row['mask_points_gis'])

                if not all(np.isfinite([coord for point in rect_coords for coord in point])):
                    continue

                poly = Polygon(rect_coords)
                if not poly.is_valid or poly.is_empty:
                    continue

                tile = str(row.get('tile_name', ''))[:50].encode('ascii', 'ignore').decode()

                shp.write({
                    'geometry': mapping(poly),
                    'properties': {
                        'latitude': float(row.get('latitude', 0)),
                        'longitude': float(row.get('longitude', 0)),
                        'class': int(row.get('class', 0)),
                        'confidence': float(row.get('confidence', 0)),
                        'tile_name': tile,
                        'area_geo': float(row.get('area_geo', 0))
                    }
                })

            except Exception as e:
                print(f"⚠️ Skipped row {idx} due to error: {e}")

    print(f"\n✅ Shapefile successfully written to: {shapefile_path}")


In [104]:
save_shapefile_from_csv(
    csv_path="annotated_output_test_2/final.csv",
    output_dir="annotated_output_test_2/annotated_output"
)


✅ Shapefile successfully written to: annotated_output_test_2/annotated_output\bbox_shapes.shp


Kept 109 polygons out of 129.
